# Lab 1 — Optimizer Benchmark trên FashionMNIST (Kaggle)

Notebook này chạy benchmark 8 optimizers × 3 LRs × 3 seeds trên MLP/CNN, tự động backup kết quả lên GitHub.

## Setup trước khi chạy (làm 1 lần)
1. Tạo GitHub **Personal Access Token** (classic, scope `repo`) tại https://github.com/settings/tokens
2. Kaggle: **Add-ons → Secrets → Add secret**, tên `GITHUB_TOKEN`, giá trị là token vừa tạo
3. Bật **GPU T4 x2** hoặc **GPU T4 x1** trong Settings (chỉ dùng 1 GPU là đủ, model nhỏ)
4. Bật **Internet** trong Settings (cần để clone repo)

> **Resume**: session chết / Stop / 12h hết → mở lại notebook, chạy lại từ đầu. Nó tự pull code mới nhất, tự bỏ qua các run đã xong, tự resume run đang dở từ checkpoint (mỗi 200 batch).

In [ ]:
import os
REPO = "https://github.com/thanh1912-ut/lab1-nhom8"
WORK = "/kaggle/working/lab1-nhom8"

# Load GITHUB_TOKEN from Kaggle Secrets
from kaggle_secrets import UserSecretsClient
os.environ["GITHUB_TOKEN"] = UserSecretsClient().get_secret("GITHUB_TOKEN")

%cd /kaggle/working
if not os.path.isdir(WORK):
    !git clone -q {REPO} lab1-nhom8
else:
    !cd lab1-nhom8 && git pull -q || true
%cd {WORK}
!pip install -q -r requirements.txt 2>&1 | tail -1

In [ ]:
# Kiểm tra GPU
import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

## Chạy benchmark
Tuần tự: MLP trước, CNN sau. Mỗi run in log kiểu YOLO (box header + epoch line + bảng metrics mỗi 5 epoch).

Có thể tách thành 2 notebook riêng (1 model mỗi notebook) để chạy song song trên 2 GPU T4.

In [ ]:
# MLP (72 runs) — Ctrl+C bất cứ lúc nào để dừng; chạy lại sẽ resume tự động
!python train_mlp.py

In [ ]:
# CNN (72 runs)
!python train_cnn.py

## Xem TensorBoard (trong lúc đang train ở cell khác)
Chạy cell này, click link `tensorboard` xuất hiện ở output.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir outputs/tensorboard --host 0.0.0.0

## Push thủ công (nếu cần) + tổng quan kết quả

In [ ]:
import sys, json
sys.path.insert(0, "src")
from benchmark import load_config
from gitbackup import backup
cfg = load_config("configs/mlp.yaml")
backup(cfg, msg="manual backup from kaggle notebook")

r = json.load(open("outputs/results.json"))
done = [h for h in r.values() if h.get("done")]
print(f"\n{len(done)} runs done.")
for m in ("mlp", "cnn"):
    rows = sorted((h for h in done if h["model"] == m),
                  key=lambda h: -(h.get("test") or {}).get("acc", 0))
    if rows:
        print(f"\n{m.upper()} top 10:")
        for h in rows[:10]:
            t = h.get("test", {})
            print(f"  {h['optimizer']:<10} lr={h['lr']:<7} s={h['seed']:<5} "
                  f"test acc={t.get('acc', 0):.4f} f1={t.get('f1', 0):.4f} "
                  f"best_ep={h['best_epoch']}")